# Forecast-Vintage Data Quality

This notebook inspects the forecast-vintage macro loader. It does not train models and it does not use the old jump-fixing or winsorized real-time panel.

Data policy:
- A forecast dated month-end `t` uses the next-month vintage/information set.
- ALFRED `component_history` is primary where available.
- Historical FRED-MD archive files are fallback only.
- Dates before the first archive vintage use the earliest archive as an explicitly tagged anchor fallback.
- Latest revised FRED-MD is never used as fallback.
- Missing cells remain `NaN` and are tagged.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.forecast_vintages import ForecastVintageMacroStore, SOURCE_MISSING

store = ForecastVintageMacroStore()
pd.set_option('display.max_columns', 20)


## Coverage And Date Mapping

In [ ]:
mapping = store.date_mapping_examples(['1990-01-31', '2018-12-31', '2025-12-31'])
display(mapping)

cache_summary = store.cache_coverage_summary()
archive_summary = store.archive_coverage_summary()
error_log = store.error_log()

overview = pd.DataFrame([
    {'metric': 'alfred_cache_series', 'value': cache_summary['series_id'].nunique()},
    {'metric': 'alfred_cache_rows', 'value': int(cache_summary['n_rows'].sum())},
    {'metric': 'alfred_obs_min', 'value': cache_summary['obs_min'].min()},
    {'metric': 'alfred_obs_max', 'value': cache_summary['obs_max'].max()},
    {'metric': 'alfred_realtime_min', 'value': cache_summary['realtime_min'].min()},
    {'metric': 'alfred_realtime_max', 'value': cache_summary['realtime_max'].max()},
    {'metric': 'archive_vintage_count', 'value': len(archive_summary)},
    {'metric': 'archive_first_vintage', 'value': archive_summary['vintage_month'].min()},
    {'metric': 'archive_last_vintage', 'value': archive_summary['vintage_month'].max()},
    {'metric': 'error_log_rows', 'value': len(error_log)},
])
display(overview)
display(cache_summary.sort_values('n_rows').head(10))


## Example Panels

In [ ]:
inspection_start = '1989-01-31'
inspection_end = '2025-12-31'
inspection_dates = pd.date_range(inspection_start, inspection_end, freq='M')

example_dates = ['1990-01-31', '2018-12-31', '2025-12-31']
panels = {
    date: store.panel_for_forecast_date(date, start=inspection_start, end=date)
    for date in example_dates
}
stitched_panel = store.row_panel_for_dates(inspection_dates, start=inspection_start)
latest_asof_panel = store.panel_for_forecast_date(inspection_end, start=inspection_start, end=inspection_end)

metadata = pd.DataFrame([panel.metadata for panel in panels.values()])
display(metadata)
display(pd.DataFrame([
    {'panel': 'stitched_forecast_rows', 'rows': stitched_panel.transformed.shape[0], 'cols': stitched_panel.transformed.shape[1]},
    {'panel': 'single_asof_full_history', 'rows': latest_asof_panel.transformed.shape[0], 'cols': latest_asof_panel.transformed.shape[1]},
]))

source_mix = []
for date, panel in panels.items():
    counts = panel.source_tags.stack().value_counts(dropna=False)
    for tag, count in counts.items():
        source_mix.append({'forecast_date': date, 'source_tag': tag, 'count': int(count)})
source_mix = pd.DataFrame(source_mix)
source_mix['share_pct'] = source_mix.groupby('forecast_date')['count'].transform(lambda s: 100 * s / s.sum()).round(2)
display(source_mix)


## Missingness

Separate four different notions of missingness:
- stitched forecast-row panel missingness;
- single as-of full-history panel missingness;
- source-level `MISSING` tags;
- transformation-induced `NaN`s where raw/source tags are present but the FRED-MD transform still yields `NaN`.

In [ ]:
inspection_panel = panels['2018-12-31']

def summarize_missingness(panel, label):
    transformed_mask = panel.transformed.isna()
    source_missing_mask = panel.source_tags.eq(SOURCE_MISSING)
    transform_only_mask = transformed_mask & ~source_missing_mask
    total_cells = int(panel.transformed.size)
    transformed_nans = int(transformed_mask.sum().sum())
    source_missing = int(source_missing_mask.sum().sum())
    transform_only_nans = int(transform_only_mask.sum().sum())
    return {
        'panel': label,
        'rows': panel.transformed.shape[0],
        'cols': panel.transformed.shape[1],
        'total_cells': total_cells,
        'transformed_nans': transformed_nans,
        'source_missing_tags': source_missing,
        'transformation_only_nans': transform_only_nans,
        'transformed_nan_share_pct': round(100 * transformed_nans / total_cells, 2),
        'source_missing_share_pct': round(100 * source_missing / total_cells, 2),
        'transform_only_share_pct': round(100 * transform_only_nans / total_cells, 2),
    }

missingness_summary = pd.DataFrame([
    summarize_missingness(stitched_panel, 'stitched_forecast_rows'),
    summarize_missingness(latest_asof_panel, 'single_asof_full_history_2025_12_31'),
]).set_index('panel')
display(missingness_summary)

stitched_missing_by_series = stitched_panel.transformed.isna().sum().sort_values(ascending=False)
stitched_missing_by_date = stitched_panel.transformed.isna().sum(axis=1)
latest_asof_missing_by_series = latest_asof_panel.transformed.isna().sum().sort_values(ascending=False)
latest_asof_missing_by_date = latest_asof_panel.transformed.isna().sum(axis=1)

display(pd.DataFrame({
    'stitched_forecast_rows': stitched_missing_by_series.head(20),
    'single_asof_full_history': latest_asof_missing_by_series.head(20),
}).fillna(0).astype(int))
display(pd.DataFrame({
    'stitched_forecast_rows': stitched_missing_by_date.tail(12),
    'single_asof_full_history': latest_asof_missing_by_date.tail(12),
}))

transformation_only_by_series = (
    (stitched_panel.transformed.isna() & ~stitched_panel.source_tags.eq(SOURCE_MISSING))
    .sum()
    .sort_values(ascending=False)
)
display(transformation_only_by_series.head(20).rename('transformation_only_nans').to_frame())

assert inspection_panel.metadata['uses_latest_revised_fred_md'] is False
assert inspection_panel.transformed.columns.is_unique
assert inspection_panel.raw.shape == inspection_panel.source_tags.shape
assert latest_asof_panel.transformed.columns.equals(stitched_panel.transformed.columns)


## Prior Jump Series Diagnostics

In [ ]:
problem_series = [
    'USTPU', 'USTRADE', 'USWTRADE', 'CPIAUCSL', 'DDURRG3M086SBEA',
    'DNDGRG3M086SBEA', 'DSERRG3M086SBEA', 'PCEPI', 'MANEMP', 'NDMANEMP'
]
available = [s for s in problem_series if s in inspection_panel.transformed.columns]

old_path = ROOT / 'data' / 'ALFRED' / 'simple_outputs' / 'realtime_final_balanced.csv'
old = pd.read_csv(old_path, index_col=0, parse_dates=True).reindex(inspection_panel.transformed.index)
old_available = [s for s in available if s in old.columns]

jump_summary = pd.DataFrame({
    'new_abs_diff_max': inspection_panel.transformed[available].diff().abs().max(),
    'old_abs_diff_max': old[old_available].diff().abs().max(),
}).sort_values('new_abs_diff_max', ascending=False)
display(jump_summary)

plot_cols = available[:6]
axes = inspection_panel.transformed[plot_cols].plot(subplots=True, figsize=(12, 8), sharex=True, title='Forecast-vintage transformed series')
plt.tight_layout()
